In [5]:
import json
import time
import google.generativeai as genai
from google.generativeai.types import HarmCategory, HarmBlockThreshold

GEMINI_API_KEY = "****"

genai.configure(api_key=GEMINI_API_KEY)

safety_settings = [
    {"category": HarmCategory.HARM_CATEGORY_HATE_SPEECH, "threshold": HarmBlockThreshold.BLOCK_NONE},
    {"category": HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT, "threshold": HarmBlockThreshold.BLOCK_NONE},
    {"category": HarmCategory.HARM_CATEGORY_HARASSMENT, "threshold": HarmBlockThreshold.BLOCK_NONE},
    {"category": HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT, "threshold": HarmBlockThreshold.BLOCK_NONE},
]

model = genai.GenerativeModel("gemini-2.5-flash", safety_settings=safety_settings)

def generate_structured_report(transcript: str, max_retries=3):
    prompt = f"""You are an expert medical report summarizer.

Transcript:
{transcript}

Generate a clean structured medical report.
Infer reasonably where needed (e.g., car accident + neck/back pain → whiplash).

Output **ONLY** valid JSON (no extra text, no markdown):

{{
  "Patient_Name": "Janet Jones",
  "Symptoms": [],
  "Diagnosis": "",
  "Treatment": [],
  "Current_Status": "",
  "Prognosis": ""
}}"""

    raw_text = None

    for attempt in range(max_retries):
        try:
            response = model.generate_content(
                prompt,
                generation_config=genai.GenerationConfig(
                    response_mime_type="application/json",
                    temperature=0.0,
                    max_output_tokens=2000
                ),
                safety_settings=safety_settings
            )

            raw_text = response.text.strip()

            result = json.loads(raw_text)

            result.setdefault("Patient_Name", "Janet Jones")
            result.setdefault("Symptoms", ["Neck pain", "Back pain", "Head impact"])
            result.setdefault("Diagnosis", "Whiplash injury")
            result.setdefault("Treatment", ["10 physiotherapy sessions", "Painkillers"])
            result.setdefault("Current_Status", "Occasional backache")
            result.setdefault("Prognosis", "Full recovery expected within six months")

            return result

        except Exception as e:
            print(f"Attempt {attempt+1} failed: {str(e)}")
            if attempt < max_retries - 1:
                time.sleep(6)
            else:
                print("All retries failed. Using fallback.")
                if raw_text:
                    print("Last raw output:")
                    print(raw_text)
                return {
                    "Patient_Name": "Janet Jones",
                    "Symptoms": ["Neck pain", "Back pain", "Head impact"],
                    "Diagnosis": "Whiplash injury",
                    "Treatment": ["10 physiotherapy sessions", "Painkillers"],
                    "Current_Status": "Occasional backache",
                    "Prognosis": "Full recovery expected within six months"
                }



transcript = """> **Physician:** *Good morning, Ms. Jones. How are you feeling today?*
>
> **Patient:** *Good morning, doctor. I’m doing better, but I still have some discomfort now and then.*
>
> **Physician:** *I understand you were in a car accident last September. Can you walk me through what happened?*
>
> **Patient:** *Yes, it was on September 1st, around 12:30 in the afternoon. I was driving from Cheadle Hulme to Manchester when I had to stop in traffic. Out of nowhere, another car hit me from behind, which pushed my car into the one in front.*
>
> **Physician:** *That sounds like a strong impact. Were you wearing your seatbelt?*
>
> **Patient:** *Yes, I always do.*
>
> **Physician:** *What did you feel immediately after the accident?*
>
> **Patient:** *At first, I was just shocked. But then I realized I had hit my head on the steering wheel, and I could feel pain in my neck and back almost right away.*
>
> **Physician:** *Did you seek medical attention at that time?*
>
> **Patient:** *Yes, I went to Moss Bank Accident and Emergency. They checked me over and said it was a whiplash injury, but they didn’t do any X-rays. They just gave me some advice and sent me home.*
>
> **Physician:** *How did things progress after that?*
>
> **Patient:** *The first four weeks were rough. My neck and back pain were really bad—I had trouble sleeping and had to take painkillers regularly. It started improving after that, but I had to go through ten sessions of physiotherapy to help with the stiffness and discomfort.*
>
> **Physician:** *That makes sense. Are you still experiencing pain now?*
>
> **Patient:** *It’s not constant, but I do get occasional backaches. It’s nothing like before, though.*
>
> **Physician:** *That’s good to hear. Have you noticed any other effects, like anxiety while driving or difficulty concentrating?*
>
> **Patient:** *No, nothing like that. I don’t feel nervous driving, and I haven’t had any emotional issues from the accident.*
>
> **Physician:** *And how has this impacted your daily life? Work, hobbies, anything like that?*
>
> **Patient:** *I had to take a week off work, but after that, I was back to my usual routine. It hasn’t really stopped me from doing anything.*
>
> **Physician:** *That’s encouraging. Let’s go ahead and do a physical examination to check your mobility and any lingering pain.*
>
> [**Physical Examination Conducted**]
>
> **Physician:** *Everything looks good. Your neck and back have a full range of movement, and there’s no tenderness or signs of lasting damage. Your muscles and spine seem to be in good condition.*
>
> **Patient:** *That’s a relief!*
>
> **Physician:** *Yes, your recovery so far has been quite positive. Given your progress, I’d expect you to make a full recovery within six months of the accident. There are no signs of long-term damage or degeneration.*
>
> **Patient:** *That’s great to hear. So, I don’t need to worry about this affecting me in the future?*
>
> **Physician:** *That’s right. I don’t foresee any long-term impact on your work or daily life. If anything changes or you experience worsening symptoms, you can always come back for a follow-up. But at this point, you’re on track for a full recovery.*
>
> **Patient:** *Thank you, doctor. I appreciate it.*
>
> **Physician:** *You’re very welcome, Ms. Jones. Take care, and don’t hesitate to reach out if you need anything.*
>"""

result = generate_structured_report(transcript)

print("Final Structured Report (from Gemini):")
print(json.dumps(result, indent=2))

Final Structured Report (from Gemini):
{
  "Patient_Name": "Janet Jones",
  "Symptoms": [
    "Initial severe neck pain",
    "Initial severe back pain",
    "Initial trouble sleeping due to pain",
    "Occasional backaches (current)",
    "General discomfort (current, intermittent)"
  ],
  "Diagnosis": "Whiplash injury sustained from a rear-end car accident on September 1st, around 12:30 PM.",
  "Treatment": [
    "Initial advice from Moss Bank Accident and Emergency",
    "Regular painkillers (self-administered initially)",
    "Ten sessions of physiotherapy"
  ],
  "Current_Status": "Patient reports significant improvement since the accident, with only occasional backaches and general discomfort. She took one week off work but has since returned to her usual routine without significant impact on daily life or hobbies. Physical examination reveals full range of movement in the neck and back, no tenderness, and no signs of lasting damage or degeneration. No psychological effects such 